In [0]:
CATALOG = 'ddmfa_catalog'
SCHEMA = 'drone_schema'
VOLUME = 'raw_data'
RAW_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}'

files = [f.path for f in dbutils.fs.ls(RAW_PATH) if not f.isDir()]
results = []

for path in sorted(files):
    if path.lower().endswith('.csv'):
        df = spark.read.option('header', True).option('inferSchema', True).csv(path)
        results.append((path.split('/')[-1], df.count(), len(df.columns), ', '.join(df.columns)))

result_df = spark.createDataFrame(results, ['file_name', 'row_count', 'column_count', 'columns'])
display(result_df.orderBy('file_name'))

file_name,row_count,column_count,columns
deliveries.csv,5000,8,"delivery_id, drone_id, source, destination, distance_km, start_time, end_time, status"
drones.csv,100,3,"drone_id, model, max_range_km"
drones_update.csv,5,3,"drone_id, model, max_range_km"
flight_logs.csv,30150,8,"log_id, drone_id, delivery_id, timestamp, battery_level, gps_signal, weather_condition, status"


# Phase 2 — Silver Layer: Cleaning, Transformation & Failure Classification

**What this notebook does:**
- Reads all 3 Bronze Delta tables
- Casts columns to correct explicit types
- Fills nulls using median/mode strategies (recorded per column)
- Derives `delivery_duration_mins`, `failure_flag`, `failure_cause`
- Applies **threshold-based failure classification** on flight logs
- Runs **SCD Type 1 MERGE** on `silver_drones` using recalibrated specs
- Logs a Data Quality record to `dq_audit_log`
- Writes 3 Silver Delta tables

**Failure classification thresholds (design decision):**
- `battery_level < 20` → `FAILED_BATTERY`
- `gps_signal < 0.30` → `FAILED_SIGNAL`
- `weather_condition in [heavy_rain, storm]` → `FAILED_WEATHER`
- Priority: BATTERY > SIGNAL > WEATHER > UNKNOWN
- Classification is applied only where `status = FAILED`

In [0]:
# ── Configuration ──────────────────────────────────────────────────────
CATALOG  = "ddmfa_catalog"
SCHEMA   = "drone_schema"
VOLUME   = "raw_data"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DB       = f"{CATALOG}.{SCHEMA}"

BATTERY_THRESHOLD       = 20.0
SIGNAL_THRESHOLD        = 0.30
WEATHER_FAIL_CONDITIONS = ["heavy_rain", "storm"]

print(f"Source (Bronze) : {DB}")
print(f"Target (Silver) : {DB}")
print(f"Thresholds      : battery < {BATTERY_THRESHOLD}, signal < {SIGNAL_THRESHOLD}")

Source (Bronze) : ddmfa_catalog.drone_schema
Target (Silver) : ddmfa_catalog.drone_schema
Thresholds      : battery < 20.0, signal < 0.3


In [0]:
from pyspark.sql.functions import (
    col, lit, when, current_timestamp,
    unix_timestamp, round as spark_round,
    to_timestamp
)
from pyspark.sql import DataFrame

# to track null fill operations across all silver tables
dq_log_rows = []

def null_fill_audit(df_before: DataFrame, df_after: DataFrame,
                    column: str, strategy: str, table: str):
    before_nulls = df_before.filter(col(column).isNull()).count()
    after_nulls  = df_after.filter(col(column).isNull()).count()
    filled       = before_nulls - after_nulls
    dq_log_rows.append((table, column, before_nulls, after_nulls, filled, strategy))
    print(f"  [{table}] {column}")
    print(f"    Nulls before : {before_nulls:,}")
    print(f"    Nulls after  : {after_nulls:,}")
    print(f"    Filled       : {filled:,}  (strategy: {strategy})")

## 1. `bronze_drones` → `silver_drones`
Type casting and metadata.
 
No nulls exist here (as confirmed in Bronze audit).

In [0]:
df_bronze_drones = spark.table(f"{DB}.bronze_drones")

df_silver_drones = (
    df_bronze_drones
    .withColumn("drone_id",       col("drone_id").cast("string"))
    .withColumn("model",          col("model").cast("string"))
    .withColumn("max_range_km",   col("max_range_km").cast("double"))
    .withColumn("processed_time", current_timestamp())
)

(
    df_silver_drones.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_drones")
)

print(f"silver_drones written — {df_silver_drones.count():,} rows")
df_silver_drones.printSchema()

silver_drones written — 100 rows
root
 |-- drone_id: string (nullable = true)
 |-- model: string (nullable = true)
 |-- max_range_km: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- processed_time: timestamp (nullable = false)



## 2. SCD Type 1 MERGE on `silver_drones`
5 drones have recalibrated `max_range_km` values in `drones_update.csv`.

SCD Type 1 - overwrite the old value, no history kept.

For handling updates in the database.

In [0]:
df_updates = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/drones_update.csv")
    .withColumn("processed_time", current_timestamp())
)

print("Incoming drone updates:")
df_updates.select("drone_id", "max_range_km").show(truncate=False)

df_updates.createOrReplaceTempView("drone_updates_view")

# list the updated_ids here
updated_ids = [r["drone_id"] for r in df_updates.collect()]

# Read BEFORE state , before MERGE runs
before_rows = (
    spark.table(f"{DB}.silver_drones")
    .filter(col("drone_id").isin(updated_ids))
    .select("drone_id", col("max_range_km").alias("max_range_km_before"))
).collect()

# Run MERGE
spark.sql(f"""
    MERGE INTO {DB}.silver_drones AS target
    USING drone_updates_view AS source
    ON target.drone_id = source.drone_id
    WHEN MATCHED THEN UPDATE SET
        target.max_range_km   = source.max_range_km,
        target.processed_time = source.processed_time
""")

# Read AFTER state
df_after = (
    spark.table(f"{DB}.silver_drones")
    .filter(col("drone_id").isin(updated_ids))
    .select("drone_id", col("max_range_km").alias("max_range_km_after"))
)

df_before = spark.createDataFrame(before_rows)
print("SCD Type 1 — Before vs After:")
df_before.join(df_after, "drone_id").show(truncate=False)
print("SCD Type 1 MERGE complete")

Incoming drone updates:
+--------+------------+
|drone_id|max_range_km|
+--------+------------+
|D038    |59.0        |
|D027    |110.1       |
|D079    |38.2        |
|D092    |47.1        |
|D050    |36.5        |
+--------+------------+

SCD Type 1 — Before vs After:
+--------+-------------------+------------------+
|drone_id|max_range_km_before|max_range_km_after|
+--------+-------------------+------------------+
|D027    |104.2              |110.1             |
|D079    |41.1               |38.2              |
|D038    |56.6               |59.0              |
|D050    |34.9               |36.5              |
|D092    |46.6               |47.1              |
+--------+-------------------+------------------+

SCD Type 1 MERGE complete


## 3. `bronze_deliveries` → `silver_deliveries`
- Fill `distance_km` nulls with **median** (50 nulls, 1.0%)
- Cast `start_time` and `end_time` to timestamp
- Derive `delivery_duration_mins` = (end_time - start_time) / 60
- Derive `failure_flag` = 1 if status = FAILED, else 0

In [0]:
df_bronze_del = spark.table(f"{DB}.bronze_deliveries")

median_distance = df_bronze_del.approxQuantile("distance_km", [0.5], 0.01)[0]
print(f"Median distance_km: {median_distance:.2f} km (used to fill nulls)")

df_filled_del = df_bronze_del.fillna({"distance_km": round(median_distance, 2)})

print("\nNull Fill Audit:")
null_fill_audit(df_bronze_del, df_filled_del, "distance_km",
                f"median ({median_distance:.2f})", "silver_deliveries")

Median distance_km: 27.13 km (used to fill nulls)

Null Fill Audit:
  [silver_deliveries] distance_km
    Nulls before : 50
    Nulls after  : 0
    Filled       : 50  (strategy: median (27.13))


In [0]:
df_silver_del = (
    df_filled_del
    .withColumn("delivery_id",  col("delivery_id").cast("string"))
    .withColumn("drone_id",     col("drone_id").cast("string"))
    .withColumn("source",       col("source").cast("string"))
    .withColumn("destination",  col("destination").cast("string"))
    .withColumn("distance_km",  col("distance_km").cast("double"))
    .withColumn("start_time",   to_timestamp(col("start_time"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("end_time",     to_timestamp(col("end_time"),   "yyyy-MM-dd HH:mm:ss"))
    .withColumn("status",       col("status").cast("string"))
    .withColumn(
        "delivery_duration_mins",
        spark_round(
            (unix_timestamp(col("end_time")) - unix_timestamp(col("start_time"))) / 60.0,
            2
        )
    )
    .withColumn(
        "failure_flag",
        when(col("status") == "FAILED", 1).otherwise(0)
    )
    .withColumn("processed_time", current_timestamp())
    .drop("ingestion_time", "source_file")
)

(
    df_silver_del.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_deliveries")
)

print(f"silver_deliveries written — {df_silver_del.count():,} rows")
df_silver_del.printSchema()

silver_deliveries written — 5,000 rows
root
 |-- delivery_id: string (nullable = true)
 |-- drone_id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- distance_km: double (nullable = false)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- delivery_duration_mins: double (nullable = true)
 |-- failure_flag: integer (nullable = false)
 |-- processed_time: timestamp (nullable = false)



In [0]:
print("delivery_duration_mins distribution:")
spark.table(f"{DB}.silver_deliveries").select("delivery_duration_mins").summary().show()

print("failure_flag distribution:")
spark.table(f"{DB}.silver_deliveries").groupBy("failure_flag").count().show()

delivery_duration_mins distribution:
+-------+----------------------+
|summary|delivery_duration_mins|
+-------+----------------------+
|  count|                  5000|
|   mean|    27.859428000000026|
| stddev|    19.408848177837207|
|    min|                   1.8|
|    25%|                 13.18|
|    50%|                 23.32|
|    75%|                 37.62|
|    max|                141.67|
+-------+----------------------+

failure_flag distribution:
+------------+-----+
|failure_flag|count|
+------------+-----+
|           0| 4093|
|           1|  907|
+------------+-----+



## 4. `bronze_flight_logs` → `silver_flight_logs`
- Fill `battery_level` nulls with **median**
- Fill `gps_signal` nulls with **median**
- Fill `weather_condition` nulls with **mode**
- Derive `failure_flag` = 1 if status = FAILED, else 0
- Derive `failure_cause` using threshold-based classification

I fill Nulls BEFORE classification because a null battery level on a FAILED log
would make it unclassifiable without filling first.

In [0]:
df_bronze_logs = spark.table(f"{DB}.bronze_flight_logs")

median_battery = df_bronze_logs.approxQuantile("battery_level", [0.5], 0.01)[0]
median_gps     = df_bronze_logs.approxQuantile("gps_signal",    [0.5], 0.01)[0]
mode_weather   = (
    df_bronze_logs
    .filter(col("weather_condition").isNotNull())
    .groupBy("weather_condition")
    .count()
    .orderBy("count", ascending=False)
    .first()[0]
)

print(f"Median battery_level   : {median_battery:.2f}%")
print(f"Median gps_signal      : {median_gps:.3f}")
print(f"Mode weather_condition : {mode_weather}")

df_filled_logs = df_bronze_logs.fillna({
    "battery_level":     round(median_battery, 2),
    "gps_signal":        round(median_gps,     3),
    "weather_condition": mode_weather
})

print("\nNull Fill Audit:")
null_fill_audit(df_bronze_logs, df_filled_logs, "battery_level",
                f"median ({median_battery:.2f})", "silver_flight_logs")
null_fill_audit(df_bronze_logs, df_filled_logs, "gps_signal",
                f"median ({median_gps:.3f})", "silver_flight_logs")
null_fill_audit(df_bronze_logs, df_filled_logs, "weather_condition",
                f"mode ({mode_weather})", "silver_flight_logs")

Median battery_level   : 80.20%
Median gps_signal      : 0.767
Mode weather_condition : clear

Null Fill Audit:
  [silver_flight_logs] battery_level
    Nulls before : 875
    Nulls after  : 0
    Filled       : 875  (strategy: median (80.20))
  [silver_flight_logs] gps_signal
    Nulls before : 609
    Nulls after  : 0
    Filled       : 609  (strategy: median (0.767))
  [silver_flight_logs] weather_condition
    Nulls before : 568
    Nulls after  : 0
    Filled       : 568  (strategy: mode (clear))


In [0]:
df_silver_logs = (
    df_filled_logs
    .withColumn("log_id",            col("log_id").cast("string"))
    .withColumn("drone_id",          col("drone_id").cast("string"))
    .withColumn("delivery_id",       col("delivery_id").cast("string"))
    .withColumn("timestamp",         to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("battery_level",     col("battery_level").cast("double"))
    .withColumn("gps_signal",        col("gps_signal").cast("double"))
    .withColumn("weather_condition", col("weather_condition").cast("string"))
    .withColumn("status",            col("status").cast("string"))
    .withColumn(
        "failure_flag",
        when(col("status") == "FAILED", 1).otherwise(0)
    )
    .withColumn(
        "failure_cause",
        when(col("failure_flag") == 0, lit(None))
        .when(col("battery_level") < BATTERY_THRESHOLD,                       lit("FAILED_BATTERY"))
        .when(col("gps_signal")    < SIGNAL_THRESHOLD,                        lit("FAILED_SIGNAL"))
        .when(col("weather_condition").isin(WEATHER_FAIL_CONDITIONS),          lit("FAILED_WEATHER"))
        .otherwise(                                                             lit("UNKNOWN_FAILURE"))
    )
    .withColumn("processed_time", current_timestamp())
    .drop("ingestion_time", "source_file")
)

(
    df_silver_logs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_flight_logs")
)

print(f"silver_flight_logs written — {df_silver_logs.count():,} rows")
df_silver_logs.printSchema()

silver_flight_logs written — 30,000 rows
root
 |-- log_id: string (nullable = true)
 |-- drone_id: string (nullable = true)
 |-- delivery_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- battery_level: double (nullable = false)
 |-- gps_signal: double (nullable = false)
 |-- weather_condition: string (nullable = false)
 |-- status: string (nullable = true)
 |-- failure_flag: integer (nullable = false)
 |-- failure_cause: string (nullable = true)
 |-- processed_time: timestamp (nullable = false)



## 5. Data Quality Audit Log

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

dq_schema = StructType([
    StructField("table_name",     StringType(),  True),
    StructField("column_name",    StringType(),  True),
    StructField("nulls_before",   IntegerType(), True),
    StructField("nulls_after",    IntegerType(), True),
    StructField("records_filled", IntegerType(), True),
    StructField("fill_strategy",  StringType(),  True),
])

df_dq_log = (
    spark.createDataFrame(dq_log_rows, schema=dq_schema)
    .withColumn("audit_time", current_timestamp())
)

(
    df_dq_log.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.dq_audit_log")
)

print("Data Quality Audit Log:")
df_dq_log.show(truncate=False)
print("dq_audit_log written")

Data Quality Audit Log:
+------------------+-----------------+------------+-----------+--------------+--------------+--------------------------+
|table_name        |column_name      |nulls_before|nulls_after|records_filled|fill_strategy |audit_time                |
+------------------+-----------------+------------+-----------+--------------+--------------+--------------------------+
|silver_deliveries |distance_km      |50          |0          |50            |median (27.13)|2026-07-18 20:24:09.766962|
|silver_flight_logs|battery_level    |875         |0          |875           |median (80.20)|2026-07-18 20:24:09.766962|
|silver_flight_logs|gps_signal       |609         |0          |609           |median (0.767)|2026-07-18 20:24:09.766962|
|silver_flight_logs|weather_condition|568         |0          |568           |mode (clear)  |2026-07-18 20:24:09.766962|
+------------------+-----------------+------------+-----------+--------------+--------------+--------------------------+

dq_audi

## 6. Validation

In [0]:
silver_tables = {
    "silver_drones"      : 100,
    "silver_deliveries"  : 5000,
    "silver_flight_logs" : 30000,
}

print("Silver Layer Row Count Validation:")
for table, expected in silver_tables.items():
    actual = spark.table(f"{DB}.{table}").count()
    status = "OK" if actual == expected else "CHECK"
    print(f"  {status}  {table:<25} {actual:>6,}  (expected {expected:,})")

Silver Layer Row Count Validation:
  OK  silver_drones                100  (expected 100)
  OK  silver_deliveries          5,000  (expected 5,000)
  OK  silver_flight_logs        30,000  (expected 30,000)


In [0]:
print("Failure Cause Distribution (silver_flight_logs — FAILED entries only):")
(
    spark.table(f"{DB}.silver_flight_logs")
    .filter(col("failure_flag") == 1)
    .groupBy("failure_cause")
    .count()
    .orderBy("count", ascending=False)
    .show()
)

total    = spark.table(f"{DB}.silver_deliveries").count()
failures = spark.table(f"{DB}.silver_deliveries").filter(col("failure_flag") == 1).count()
print(f"Delivery Failure Rate:")
print(f"  Failed     : {failures:,}  ({failures/total*100:.1f}%)")
print(f"  Successful : {total - failures:,}  ({(total-failures)/total*100:.1f}%)")

Failure Cause Distribution (silver_flight_logs — FAILED entries only):
+---------------+-----+
|  failure_cause|count|
+---------------+-----+
| FAILED_BATTERY|  335|
|  FAILED_SIGNAL|  282|
| FAILED_WEATHER|  203|
|UNKNOWN_FAILURE|   87|
+---------------+-----+

Delivery Failure Rate:
  Failed     : 907  (18.1%)
  Successful : 4,093  (81.9%)


In [0]:
print("silver_drones (3 rows):")
spark.table(f"{DB}.silver_drones").show(3, truncate=False)

print("silver_deliveries (3 rows):")
spark.table(f"{DB}.silver_deliveries").show(3, truncate=False)

print("silver_flight_logs — FAILED entries (5 rows):")
(
    spark.table(f"{DB}.silver_flight_logs")
    .filter(col("failure_flag") == 1)
    .select("log_id", "drone_id", "delivery_id", "battery_level",
            "gps_signal", "weather_condition", "failure_flag", "failure_cause")
    .show(5, truncate=False)
)

silver_drones (3 rows):
+--------+---------+------------+--------------------------+-----------+--------------------------+
|drone_id|model    |max_range_km|ingestion_time            |source_file|processed_time            |
+--------+---------+------------+--------------------------+-----------+--------------------------+
|D021    |DJI-X500 |66.1        |2026-07-18 20:22:19.269954|drones.csv |2026-07-18 20:23:42.404731|
|D047    |DJI-X500 |60.0        |2026-07-18 20:22:19.269954|drones.csv |2026-07-18 20:23:42.404731|
|D063    |Skydio-D2|70.9        |2026-07-18 20:22:19.269954|drones.csv |2026-07-18 20:23:42.404731|
+--------+---------+------------+--------------------------+-----------+--------------------------+
only showing top 3 rows
silver_deliveries (3 rows):
+-----------+--------+-----------+-----------+-----------+-------------------+-------------------+-------+----------------------+------------+--------------------------+
|delivery_id|drone_id|source     |destination|distance